# Maximum useful range from ray-intersection geometry

Paper reference: §3.2 (System Model).

The closed-form depth estimator (Equation 5 in the paper) has a
denominator $\|\mathbf{r}\|^2 - (\boldsymbol{\alpha}^\top \mathbf{r})^2$
that vanishes as the camera ray and laser ray become collinear — the
configuration in which depth is unobservable.

Given the reference laser pose and a numerical tolerance $\epsilon$ on
that denominator, this notebook solves the quadratic for the range at
which the denominator crosses $\epsilon$. The result bounds the useful
operating distance of FishCamera for a given mount geometry and is the
basis of the "2–5 m" range quoted in Table 1 and throughout the paper.


In [ ]:
import numpy as np

In [ ]:
EPSILON = 1e-4

$\epsilon$ sets the minimum allowed value of the reconstruction denominator $\|\mathbf{r}\|^2 - (\boldsymbol{\alpha}^\top \mathbf{r})^2$ (normalized form). Below this, the $\lambda_2$ solve of Equation 5 is too close to the singular configuration (camera ray parallel to laser ray) to trust.

In [ ]:
laser_position = np.array([-0.04, -0.11, 0])
laser_direction = np.array([1e-10, 1e-10, 1])

laser_position, laser_direction

(array([-0.04, -0.11,  0.  ]), array([1.e-10, 1.e-10, 1.e+00]))

**Reference laser pose** — 4 cm lateral and 11 cm vertical offset from the camera focal point, pointing essentially along the optical axis (tiny $\epsilon$ in $x$/$y$ avoids exact parallelism; a laser *exactly* parallel to the camera axis has no parallax and yields infinite unobservability range).

In [ ]:
s0 = laser_direction @ laser_position
sv = laser_direction @ laser_direction
n0 = laser_position @ laser_position
nv = 2 * (laser_direction @ laser_position)
vv = laser_direction @ laser_direction

s0, sv, n0, nv

(np.float64(-1.5e-11), np.float64(1.0), np.float64(0.0137), np.float64(-3e-11))

Precompute dot products used by the quadratic. Setting the normalized denominator equal to $(1-\epsilon)$ along the laser ray $\mathbf{P}(t) = \boldsymbol{\ell} + t\boldsymbol{\alpha}$ gives an equation of the form $At^2 + Bt + C = 0$ with these coefficients.

In [ ]:
A = (1 - EPSILON) * sv - sv ** 2
B = (1 - EPSILON) * nv - 2 * s0 * sv
C = (1 - EPSILON) * n0 - s0 ** 2

A, B, C

(np.float64(-9.999999999998899e-05),
 np.float64(2.9999999999998862e-15),
 np.float64(0.01369863))

In [ ]:
t1 = (-B + np.sqrt(B**2 - 4 * A * C)) / (2 * A)
t2 = (-B - np.sqrt(B**2 - 4 * A * C)) / (2 * A)

t1, t2

(np.float64(-11.704114661078124), np.float64(11.704114661108127))

**Solve the quadratic.** The roots bracket the region where the estimator is stable — one is negative (behind the camera, unphysical), one is positive. Since the laser points along $+z$, the next cell converts $t$ to metric depth $z$.

In [ ]:
z1 = t1 * laser_direction[2] + laser_position[2]
z2 = t2 * laser_direction[2] + laser_position[2]

z1, z2

(np.float64(-11.704114661078124), np.float64(11.704114661108127))

**Maximum useful range.** At $\epsilon = 10^{-4}$, the estimator becomes ill-conditioned around $|z| \approx 11.7$ m — well beyond the paper's operating envelope of 2–5 m (Table 1). Tighter tolerances shrink this range; `notebooks/reconstruction/known_calibration.ipynb` and `calculated_calibration.ipynb` explore the accuracy-vs-yield tradeoff for different $\epsilon$ gating thresholds.